# 01 — Diagnosis Kualitas Data
### Milestone M2 · Exploratory Data Analysis (KMTI21133)

| | |
|---|---|
| **Nama tim** | _(isi)_ |
| **Anggota** | _(isi)_ |
| **Topik proyek** | _(isi)_ |
| **Tanggal** | _(isi)_ |

**Notebook ini mengisi:** dimensi **Kelengkapan diagnosis kualitas data** (bobot 25) pada Rubrik M2.

Tujuan notebook ini **hanya mendiagnosis** — belum memperbaiki apa pun. Pembersihan
dikerjakan di `02_cleaning.ipynb`. Memisahkan keduanya penting supaya temuan dapat
diperiksa ulang tanpa tercampur dengan perubahan yang sudah terlanjur dilakukan.

Periksa **kelima** dimensi di bawah. Memeriksa hanya nilai kosong dan duplikat membuat
dimensi ini bernilai 2 ke bawah.

---
> **Sebelum mengumpulkan:** jalankan **Kernel → Restart & Run All**. Notebook yang tidak dapat
> berjalan dari awal sampai akhir dianggap tidak memenuhi dimensi *ketepatan implementasi*.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 60)
plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 10,
    'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.grid': True, 'grid.alpha': 0.25,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})

DATA  = '#0E7490'   # warna utama
SOROT = '#C2410C'   # penyorot
REDUP = '#5B6672'

# Path relatif terhadap folder notebooks/ -- JANGAN diganti menjadi path absolut.
AKAR      = Path('..')
RAW       = AKAR / 'data' / 'raw'
PROCESSED = AKAR / 'data' / 'processed'
FIGURES   = AKAR / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

## 0. Muat data mentah

Baca dari `data/raw`. Berkas ini **tidak pernah diubah** sepanjang proyek.

In [ ]:
BERKAS = RAW / 'NAMA_BERKAS_ANDA.csv'     # TODO: ganti nama berkas

assert BERKAS.exists(), (
    f'Berkas tidak ditemukan: {BERKAS}\n'
    'Periksa apakah data sudah diletakkan di data/raw dan namanya sudah benar.'
)
df = pd.read_csv(BERKAS)
print(f'{df.shape[0]} baris x {df.shape[1]} kolom')
df.head()

In [ ]:
df.info()

## 1. Kelengkapan — nilai yang hilang

Jangan berhenti pada jumlahnya. Yang dinilai adalah apakah Anda memeriksa **polanya**:
apakah kekosongan berkaitan dengan kolom lain?

In [ ]:
kosong = pd.DataFrame({
    'jumlah' : df.isna().sum(),
    'persen' : (df.isna().mean() * 100).round(2),
})
kosong = kosong[kosong.jumlah > 0].sort_values('jumlah', ascending=False)
print(kosong.to_string() if len(kosong) else 'Tidak ada nilai kosong.')

In [ ]:
# Pola kekosongan: apakah baris yang kosong pada KOLOM_X berbeda dari yang terisi?
KOLOM_X = '____'        # TODO: kolom yang banyak kosongnya
PEMBANDING = '____'     # TODO: kolom numerik lain yang Anda duga berkaitan

# pembanding = df.groupby(df[KOLOM_X].isna())[PEMBANDING].agg(['count','mean']).round(2)
# pembanding.index = ['terisi', 'kosong']
# print(pembanding.to_string())

**Tulis temuan Anda di sini.** Bila rata-rata kedua kelompok jauh berbeda, kekosongannya
**tidak** acak — catat itu, karena akan menentukan penanganan di `02_cleaning.ipynb`.

> _(temuan Anda)_

## 2. Keunikan — baris kembar

In [ ]:
print('Duplikat seluruh kolom :', df.duplicated().sum())

KUNCI = ['____']        # TODO: kolom yang seharusnya unik, misalnya id transaksi
# print('Duplikat pada kolom kunci:', df.duplicated(subset=KUNCI).sum())

## 3. Konsistensi — penulisan kategori

Beda kapitalisasi dan spasi di ujung membuat satu kategori terpecah menjadi beberapa.

In [ ]:
kategorikal = df.select_dtypes(include=['object', 'string']).columns.tolist()

for k in kategorikal:
    n_mentah  = df[k].nunique()
    n_rapi    = df[k].astype(str).str.strip().str.lower().nunique()
    tanda = '  <-- PERIKSA' if n_rapi < n_mentah else ''
    print(f'{k:24s} unik: {n_mentah:4d}   setelah dirapikan: {n_rapi:4d}{tanda}')

In [ ]:
KOLOM = '____'          # TODO: kolom yang ditandai di atas
# print(sorted(df[KOLOM].dropna().unique()))

## 4. Validitas — nilai yang mustahil dan tipe data

Dua hal: apakah tipe datanya benar, dan apakah nilainya mungkin terjadi di dunia nyata.

In [ ]:
print('TIPE DATA YANG PERLU DIPERIKSA')
for k in df.columns:
    if df[k].dtype == object:
        contoh = df[k].dropna().iloc[0] if df[k].notna().any() else ''
        print(f'  {k:24s} object   contoh: {contoh!r}')

In [ ]:
# Rentang nilai numerik -- cari nilai yang tidak mungkin (negatif, nol, terlalu besar)
num = df.select_dtypes('number')
print(num.describe().T[['min','25%','50%','75%','max']].round(2).to_string())

In [ ]:
# TODO: tulis pemeriksaan untuk setiap nilai mustahil yang Anda temukan
# contoh:
# print('durasi negatif  :', (df['durasi'] < 0).sum())
# print('umur > 120      :', (df['umur'] > 120).sum())

> **Kolom tanggal:** bila format penulisannya hari/bulan/tahun, `pd.to_datetime` secara
> bawaan akan membacanya sebagai bulan/hari/tahun dan **tidak memunculkan pesan kesalahan**.
> Setiap tanggal yang harinya ≤ 12 akan tertukar secara diam-diam. Gunakan
> `format='%d/%m/%Y'`, lalu periksa `.min()`, `.max()`, dan jumlah bulan yang muncul.

## 5. Akurasi — konsistensi antarkolom

Dimensi yang paling sering dilewatkan. Cari hubungan yang **harus** berlaku di data Anda,
lalu uji apakah benar-benar berlaku.

In [ ]:
# contoh: total seharusnya = jumlah x harga_satuan
# seharusnya = df['jumlah'] * df['harga_satuan']
# selisih    = (df['total'] - seharusnya).abs()
# print('baris yang tidak cocok:', (selisih > 1).sum())

# TODO: tulis minimal satu pemeriksaan antarkolom untuk data Anda

## 6. Ringkasan temuan

Isi tabel ini. Tabel inilah yang disalin ke Data Quality Report dan menjadi dasar
urutan langkah di `02_cleaning.ipynb`.

| # | Masalah | Dimensi | Jumlah / bukti | Rencana penanganan |
|---|---------|---------|----------------|--------------------|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |

## Checklist sebelum mengumpulkan

- [ ] Kelima dimensi diperiksa, bukan hanya nilai kosong dan duplikat.
- [ ] Pola kekosongan diperiksa, bukan sekadar jumlahnya.
- [ ] Minimal satu pemeriksaan konsistensi antarkolom ditulis.
- [ ] Tabel ringkasan temuan terisi lengkap.
- [ ] `data/raw` tidak berubah — notebook ini hanya membaca.
- [ ] Restart & Run All berjalan tanpa galat.